In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
# Splitting into Training and Testing set
X = pd.read_csv("../data/processed/cicids2017_features_reduced.csv")
y = pd.read_csv("../data/processed/cicids2017_labels.csv").squeeze()

print(X.shape, y.shape)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Train:", X_train.shape, "Test:", X_test.shape)


(610492, 46) (610492,)
Train: (488393, 46) Test: (122099, 46)


In [3]:
# Scale features

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [4]:
# Set up evaluation

from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix)

import time
def evaluate_models(name, model, X_test, y_test):
    start = time.time()
    y_pred = model.predict(X_test)
    inference_time = time.time()-start

    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else y_pred

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    fpr = fp / (fp + tn)

    print(f"\n--- {name} ---")
    print(f"Accuracy:  {accuracy_score(y_test, y_pred): .4f}")
    print(f"Precision: {precision_score(y_test, y_pred): .4f}")
    print(f"Recall:    {recall_score(y_test, y_pred): .4f}")
    print(f"F1-score:  {f1_score(y_test, y_pred): .4f}")
    print(f"ROC_AUC:   {roc_auc_score(y_test, y_proba): .4f}")
    print(f"FPR:       {fpr:.4f}")
    print(f"Inference time({len(X_test)} samples:) {inference_time: .4f}s")

    return{
        "model": name, "accuracy":accuracy_score(y_test, y_pred), 
        "precision": precision_score(y_test, y_pred), "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred), "roc-auc": roc_auc_score(y_test, y_proba),
        "fpr": fpr, "inference_time": inference_time
    }

results=[]

In [5]:
# Train Logictic Regression
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(C=10, solver='lbfgs', max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)
results.append(evaluate_models("LogisticRegression", lr, X_test_scaled, y_test))


--- LogisticRegression ---
Accuracy:   0.9908
Precision:  0.9877
Recall:     0.9833
F1-score:   0.9855
ROC_AUC:    0.9994
FPR:       0.0057
Inference time(122099 samples:)  0.0126s


In [7]:
# Train KNN
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
knn.fit(X_train_scaled, y_train)
results.append(evaluate_models("KNN", knn, X_test_scaled, y_test))


--- KNN ---
Accuracy:   0.9990
Precision:  0.9979
Recall:     0.9989
F1-score:   0.9984
ROC_AUC:    0.9990
FPR:       0.0010
Inference time(122099 samples:)  120.5943s


In [6]:
# Train Decision Tree
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(max_depth=10, min_samples_split=10, criterion='gini', random_state=42)
dt.fit(X_train_scaled, y_train)
results.append(evaluate_models("Decision Tree", dt, X_test_scaled, y_test))


--- Decision Tree ---
Accuracy:   0.9980
Precision:  0.9977
Recall:     0.9961
F1-score:   0.9969
ROC_AUC:    0.9975
FPR:       0.0011
Inference time(122099 samples:)  0.0188s


In [9]:
# Compare all three in a table

results_df = pd.DataFrame(results)
print(results_df)
results_df.to_csv("../data/processed/baseline_results.csv", index=False)

                model  accuracy  precision    recall        f1   roc-auc  \
0       Decision Tree  0.998043   0.997725  0.996103  0.996914  0.997524   
1                 KNN  0.998993   0.997886  0.998942  0.998414  0.998979   
2  LogisticRegression  0.990803   0.987687  0.983278  0.985478  0.988789   

        fpr  inference_time  
0  0.001056        0.018807  
1  0.000984      120.594347  
2  0.005699        0.010434  


In [15]:
print("Train accuracy:", dt.score(X_train_scaled, y_train))
print("Test accuracy:", dt.score(X_test_scaled, y_test))

Train accuracy: 0.9978889951330179
Test accuracy: 0.9980425720112368


In [10]:
import joblib
joblib.dump(dt, "../models/decision_tree.pkl")

['../models/decision_tree.pkl']